In [38]:
from pathlib import Path
import sys
import subprocess

import pandas as pd
import yaml

from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split


WORK = Path("work")
DATA = Path("..") / "data" / "delivery_times.csv"

print("Python:", sys.version)
print("WORK:", WORK.resolve())
print("DATA:", DATA.resolve())
print("DATA exists:", DATA.exists())

Python: 3.11.9 (tags/v3.11.9:de54cf5, Apr  2 2024, 10:12:12) [MSC v.1938 64 bit (AMD64)]
WORK: C:\Users\charu\Downloads\P05-config-tests\P05-config-tests\work
DATA: C:\Users\charu\Downloads\P05-config-tests\data\delivery_times.csv
DATA exists: True


In [39]:
with open(WORK / "config.yaml", "r", encoding="utf-8") as f:
    config = yaml.safe_load(f)

print(config)

{'data': {'path': '../data/delivery_times.csv', 'features': ['distance_km', 'prep_time_min', 'traffic_level', 'rain'], 'target': 'delivery_min'}, 'split': {'test_size': 0.2, 'seed': 42}, 'model': {'name': 'linear_regression'}, 'training': {'max_rows': 300}}


In [40]:
config["training"] = {
    "max_rows": 300
}

print("Training configuration:")
print(config["training"])

Training configuration:
{'max_rows': 300}


In [41]:
with open(WORK / "config.yaml", "w", encoding="utf-8") as f:
    yaml.safe_dump(config, f, sort_keys=False)

print("config.yaml updated successfully.")

config.yaml updated successfully.


In [42]:
with open(WORK / "config.yaml", "r", encoding="utf-8") as f:
    config = yaml.safe_load(f)

print("max_rows =", config["training"]["max_rows"])

max_rows = 300


In [43]:
def train_limited(config, data_path):
    orders = pd.read_csv(data_path)

    orders = orders.head(
        config["training"]["max_rows"]
    )

    X = orders[config["data"]["features"]]
    y = orders[config["data"]["target"]]

    X_tr, X_te, y_tr, y_te = train_test_split(
        X,
        y,
        test_size=config["split"]["test_size"],
        random_state=config["split"]["seed"]
    )

    model = LinearRegression()
    model.fit(X_tr, y_tr)

    predictions = model.predict(X_te)

    return mean_absolute_error(
        y_te,
        predictions
    )

In [44]:
T1_mae = train_limited(config, DATA)

print("MAE on the first 300 orders:", T1_mae)

MAE on the first 300 orders: 2.1781751077589746


In [45]:
orders = pd.read_csv(DATA)

print("Total dataset rows:", len(orders))
print("Rows used by T1:", len(
    orders.head(config["training"]["max_rows"])
))

Total dataset rows: 600
Rows used by T1: 300


In [46]:
def run_pytest(*extra):
    done = subprocess.run(
        [
            sys.executable,
            "-m",
            "pytest",
            "-v",
            *extra
        ],
        cwd=WORK,
        capture_output=True,
        text=True,
    )

    print(done.stdout[-2500:])

    if done.stderr:
        print(done.stderr)

    print(
        "exit code:",
        done.returncode,
        "(0 means every test passed)"
    )

    return done

In [47]:
speed_test_py = '''"My own tests for minutes_per_km."

import pytest

from orders import minutes_per_km


def test_slow_delivery():
    assert minutes_per_km(60, 6) == 10.0


def test_negative_distance_raises():
    with pytest.raises(ValueError):
        minutes_per_km(30, -3)
'''

(WORK / "test_speed.py").write_text(
    speed_test_py,
    encoding="utf-8"
)

print("test_speed.py created successfully.")

test_speed.py created successfully.


In [48]:
print(
    (WORK / "test_speed.py").read_text(
        encoding="utf-8"
    )
)

"My own tests for minutes_per_km."

import pytest

from orders import minutes_per_km


def test_slow_delivery():
    assert minutes_per_km(60, 6) == 10.0


def test_negative_distance_raises():
    with pytest.raises(ValueError):
        minutes_per_km(30, -3)



In [49]:
T2_run = run_pytest("-q")

============================= test session starts =============================
platform win32 -- Python 3.11.9, pytest-9.1.1, pluggy-1.6.0
rootdir: C:\Users\charu\Downloads\P05-config-tests\P05-config-tests\work
plugins: anyio-4.14.2
collected 10 items

test_model.py ....                                                       [ 40%]
test_orders.py ....                                                      [ 80%]
test_speed.py ..                                                         [100%]

============================= 10 passed in 0.05s ==============================

exit code: 0 (0 means every test passed)


In [50]:
test_model_path = WORK / "test_model.py"

print(
    test_model_path.read_text(
        encoding="utf-8"
    )
)

"Behaviour tests for the trained model."

import pandas as pd


def make_order(distance_km=5.0, prep_time_min=20,
               traffic_level=2, rain=0):
    return pd.DataFrame([{
        "distance_km": distance_km,
        "prep_time_min": prep_time_min,
        "traffic_level": traffic_level,
        "rain": rain,
    }])


def test_rain_never_makes_delivery_faster(trained_model):
    dry = trained_model.predict(make_order(rain=0))[0]
    wet = trained_model.predict(make_order(rain=1))[0]
    assert wet >= dry


def test_further_is_never_faster(trained_model):
    near = trained_model.predict(make_order(distance_km=2))[0]
    far = trained_model.predict(make_order(distance_km=12))[0]
    assert far > near


def test_error_is_better_than_guessing(model_mae, baseline_mae):
    assert model_mae < baseline_mae


def test_heavier_traffic_is_never_faster(trained_model):
    light = trained_model.predict(
        make_order(traffic_level=1)
    )[0]

    heavy = trained_model.predict(
   

In [51]:
extra_test = '''

def test_heavier_traffic_is_never_faster(trained_model):
    light = trained_model.predict(
        make_order(traffic_level=1)
    )[0]

    heavy = trained_model.predict(
        make_order(traffic_level=3)
    )[0]

    assert heavy >= light
'''

existing_test_model = test_model_path.read_text(
    encoding="utf-8"
)

if "def test_heavier_traffic_is_never_faster(trained_model)" not in existing_test_model:
    with open(test_model_path, "a", encoding="utf-8") as fh:
        fh.write(extra_test)
    print("T3 test appended successfully.")
else:
    print("T3 test already exists. No duplicate added.")

T3 test already exists. No duplicate added.


In [52]:
test_model_code = test_model_path.read_text(
    encoding="utf-8"
)

print(test_model_code)

"Behaviour tests for the trained model."

import pandas as pd


def make_order(distance_km=5.0, prep_time_min=20,
               traffic_level=2, rain=0):
    return pd.DataFrame([{
        "distance_km": distance_km,
        "prep_time_min": prep_time_min,
        "traffic_level": traffic_level,
        "rain": rain,
    }])


def test_rain_never_makes_delivery_faster(trained_model):
    dry = trained_model.predict(make_order(rain=0))[0]
    wet = trained_model.predict(make_order(rain=1))[0]
    assert wet >= dry


def test_further_is_never_faster(trained_model):
    near = trained_model.predict(make_order(distance_km=2))[0]
    far = trained_model.predict(make_order(distance_km=12))[0]
    assert far > near


def test_error_is_better_than_guessing(model_mae, baseline_mae):
    assert model_mae < baseline_mae


def test_heavier_traffic_is_never_faster(trained_model):
    light = trained_model.predict(
        make_order(traffic_level=1)
    )[0]

    heavy = trained_model.predict(
   

In [53]:
T3_run = run_pytest("-q")

============================= test session starts =============================
platform win32 -- Python 3.11.9, pytest-9.1.1, pluggy-1.6.0
rootdir: C:\Users\charu\Downloads\P05-config-tests\P05-config-tests\work
plugins: anyio-4.14.2
collected 10 items

test_model.py ....                                                       [ 40%]
test_orders.py ....                                                      [ 80%]
test_speed.py ..                                                         [100%]

============================= 10 passed in 0.05s ==============================

exit code: 0 (0 means every test passed)


In [54]:
_results = []


def _check(label, fn):
    try:
        ok = bool(fn())
    except Exception:
        ok = False

    _results.append((label, ok))


_check(
    'T1 | config has a training section with max_rows 300',
    lambda: config['training']['max_rows'] == 300
)

_check(
    'T1 | T1_mae is a sensible error in minutes',
    lambda: 0 < float(T1_mae) < 15
)

_check(
    'T1 | train_limited really used only 300 rows',
    lambda:
        abs(
            float(
                train_limited(
                    {
                        **config,
                        'training': {
                            'max_rows': 300
                        }
                    },
                    DATA
                )
            ) - float(T1_mae)
        ) < 1e-9
        and
        abs(
            float(
                train_limited(
                    {
                        **config,
                        'training': {
                            'max_rows': 600
                        }
                    },
                    DATA
                )
            ) - float(T1_mae)
        ) > 1e-9
)

_check(
    'T2 | test_speed.py exists and names both tests',
    lambda:
        'def test_slow_delivery'
        in (WORK / 'test_speed.py').read_text(
            encoding='utf-8'
        )
        and
        'def test_negative_distance_raises'
        in (WORK / 'test_speed.py').read_text(
            encoding='utf-8'
        )
)

_check(
    'T2 | both tests really assert something',
    lambda:
        'assert'
        in (WORK / 'test_speed.py').read_text(
            encoding='utf-8'
        )
        and
        'pytest.raises'
        in (WORK / 'test_speed.py').read_text(
            encoding='utf-8'
        )
)

_check(
    'T2 | the whole suite passes (exit code 0)',
    lambda: T2_run.returncode == 0
)

_check(
    'T2 | at least 9 tests were collected',
    lambda:
        any(
            str(n) + ' passed'
            in T2_run.stdout
            for n in range(9, 40)
        )
)

_check(
    'T3 | the new test asserts something and uses the fixture',
    lambda:
        'test_heavier_traffic_is_never_faster(trained_model)'
        in (WORK / 'test_model.py').read_text(
            encoding='utf-8'
        )
        and
        (
            (WORK / 'test_model.py').read_text(
                encoding='utf-8'
            )
            .rsplit('def test_heavier', 1)[1]
            .count('assert')
            >= 1
        )
)

_check(
    'T3 | the whole suite still passes',
    lambda: T3_run.returncode == 0
)


print("==================================================================")
print("SELF-CHECK   Practical 05 --- Settings in a File, Bugs Caught by a Robot")
print("==================================================================")

for _label, _ok in _results:
    print(
        f"  [{'PASS' if _ok else 'FAIL'}]  {_label}"
    )

print("------------------------------------------------------------------")

_passed = sum(
    1
    for _, _ok in _results
    if _ok
)

print(
    f"  {_passed} of {len(_results)} checks passed"
)

print("==================================================================")

if _passed == len(_results):
    print("Well done. Save the notebook and submit it.")
else:
    print("Read the FAIL lines above, fix those tasks, run this cell again.")

SELF-CHECK   Practical 05 --- Settings in a File, Bugs Caught by a Robot
  [PASS]  T1 | config has a training section with max_rows 300
  [PASS]  T1 | T1_mae is a sensible error in minutes
  [PASS]  T1 | train_limited really used only 300 rows
  [PASS]  T2 | test_speed.py exists and names both tests
  [PASS]  T2 | both tests really assert something
  [PASS]  T2 | the whole suite passes (exit code 0)
  [PASS]  T2 | at least 9 tests were collected
  [PASS]  T3 | the new test asserts something and uses the fixture
  [PASS]  T3 | the whole suite still passes
------------------------------------------------------------------
  9 of 9 checks passed
Well done. Save the notebook and submit it.
